# Mini-ColBERTv2 on Multiple BEIR Datasets (Colab)

Chay pipeline base -> cad -> evaluation tren 4 tap BEIR de kiem chung tinh tong quat:

| Dataset | Docs | Queries | Domain |
|---|---|---|---|
| NFCorpus | 3.6k | 323 | Y sinh |
| SCIDOCS | 25k | 1000 | Khoa hoc |
| FiQA-2018 | 57k | 648 | Tai chinh |
| TREC-COVID | 171k | 50 | Y khoa |

Moi dataset chay doc lap, output luu trong `outputs/<dataset>/`.

In [ ]:
# 1. Clone repo
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git
%cd colbert-hoangnd

In [ ]:
# 2. Install dependencies (pin transformers<5 to stay compatible with colbert-ai)
!pip install -q "transformers==4.57.6" "colbert-ai==0.2.22" scikit-learn datasets beir tqdm numpy ninja

In [ ]:
# 3. Ensure ninja is on PATH for colbert-ai C++ extension compilation
import os, sys
os.environ['PATH'] += ':' + os.path.join(sys.prefix, 'bin')
!ninja --version

In [ ]:
# 4. Detect device
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# 5. Run full pipeline on all 4 datasets (base -> cad -> evaluate per dataset)
# Note: TREC-COVID is the largest (171k docs); skip it first if you want a quick smoke test.
DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'trec-covid']

for ds in DATASETS:
    out = f'outputs/{ds}'
    print(f'\n==================== {ds} ====================')
    !python run_base.py --dataset {ds} --output_dir {out} --K 8192 --n_samples 200000 --ncells 4 --candidate_topn 200 --topk 100 --device {DEVICE} --doc_batch_size 64 --query_batch_size 64
    !python run_cad.py --dataset {ds} --output_dir {out} --topk 100
    !python run_evaluate.py --output_dir {out} --qrels {out}/qrels_test.json

In [ ]:
# 6. Aggregate results across datasets into one comparison table
import json
from pathlib import Path
from evaluation.metrics import evaluate, load_json

VARIANTS = ['original', 'idf', 'adaptive', 'cad']
rows = {}
for ds in DATASETS:
    out = Path(f'outputs/{ds}')
    qrels = load_json(out / 'qrels_test.json')
    for name in VARIANTS:
        path = out / f'rankings_{name}.json'
        if not path.exists():
            continue
        metrics = evaluate(load_json(path), qrels)
        rows[(ds, name)] = metrics

print(f"{'Dataset':<14} {'Method':<10} {'nDCG@10':>8} {'MRR@10':>8} {'Recall@100':>11}")
print('-' * 55)
for ds in DATASETS:
    for name in VARIANTS:
        m = rows.get((ds, name))
        if m:
            print(f"{ds:<14} {name:<10} {m['nDCG@10']:>8.4f} {m['MRR@10']:>8.4f} {m['Recall@100']:>11.4f}")
    print('-' * 55)

In [ ]:
# 7. (Optional) Grid-search beta for CAD on each dataset
# for ds in DATASETS:
#     out = f'outputs/{ds}'
#     !python -m cad.tune_cad --candidates {out}/base_candidates.pt --query_emb {out}/query_embeddings.pt --idf {out}/idf.pt --qrels {out}/qrels_test.json --output_dir {out} --topk 100
pass